In [1]:
from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm

from cdvae.common.utils import PROJECT_ROOT
from cdvae.common.data_utils import (
    EPSILON, cart_to_frac_coords, mard, lengths_angles_to_volume,
    frac_to_cart_coords, min_distance_sqr_pbc)
from cdvae.pl_modules.embeddings import MAX_ATOMIC_NUM
from cdvae.pl_modules.embeddings import KHOT_EMBEDDINGS

#added by Tsach
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
#import Batch
from torch_geometric.data import Batch
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
torch.set_printoptions(threshold=50000) # use this if you want to print the entire tensor
import matplotlib.pyplot as plt


In [2]:
 import time
import argparse
import torch

from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
from types import SimpleNamespace
from torch_geometric.data import Batch
from torch_geometric.data.separate import separate

#import a library that allows you to reload a module
from importlib import reload

from eval_utils import load_model

all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

#my code 
list_of_idxs = []
list_of_batchs = []


In [3]:
model_path = Path("/home/gridsan/tmackey/hydra/singlerun/2023-12-22/ogCDVAE_mp_20_graph_preloading_50u")
model, test_loader, cfg = load_model(model_path, True)

loader = test_loader

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


using 9047 rows given a train_fraction of 1


  0%|          | 0/9047 [00:00<?, ?it/s]

CrystDataModule(self.datasets={'train': {'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy train', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/train.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30, 'train_fraction': 1}, 'val': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy val', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/val.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}], 'test': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy test', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/test.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/torch_geometric/deprecation.py:13: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [4]:
from pathlib import Path
from typing import List

import hydra
import numpy as np
import torch
import omegaconf
import pytorch_lightning as pl
from hydra.core.hydra_config import HydraConfig
from omegaconf import DictConfig, OmegaConf
from pytorch_lightning import seed_everything, Callback
from pytorch_lightning.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
)
from pytorch_lightning.loggers import WandbLogger

from cdvae.common.utils import log_hyperparameters, PROJECT_ROOT

In [5]:
# Assuming PROJECT_ROOT and config_path are defined
config_path = str("conf")  # config directory

import hydra
from hydra.experimental import initialize, compose
from omegaconf import OmegaConf

# Initialize Hydra
with initialize(config_path=config_path):
    #overrides to implement: ['data=perov', 'expname=perov_multiGPU_v5']
    cfg = compose(config_name="default", overrides=["data=mp_20_dm", "expname=mp_20_dm_demo"])

    # Print the configuration
    print(OmegaConf.to_yaml(cfg))


/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/initialize.py:35: UserWarning: hydra.experimental.initialize() is no longer experimental. Use hydra.initialize()
  warnings.warn(


data:
  root_path: ${oc.env:PROJECT_ROOT}/data/mp_20_dm
  prop: formation_energy_per_atom
  num_targets: 1
  niggli: true
  primitive: false
  graph_method: crystalnn
  lattice_scale_method: scale_length
  preprocess_workers: 30
  readout: mean
  max_atoms: 50
  otf_graph: false
  eval_model_name: mp20_dm
  train_max_epochs: 1000
  early_stopping_patience: 100000
  teacher_forcing_max_epoch: 500
  datamodule:
    _target_: cdvae.pl_data.datamodule.CrystDataModule
    datasets:
      train:
        _target_: cdvae.pl_data.dataset.CrystDataset
        name: Formation energy train
        path: ${data.root_path}/train.csv
        prop: ${data.prop}
        niggli: ${data.niggli}
        primitive: ${data.primitive}
        graph_method: ${data.graph_method}
        lattice_scale_method: ${data.lattice_scale_method}
        preprocess_workers: ${data.preprocess_workers}
        train_fraction: 1
      val:
      - _target_: cdvae.pl_data.dataset.CrystDataset
        name: Formation energy 

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


In [6]:
# Instantiate datamodule
hydra.utils.log.info(f"Instantiating <{cfg.data.datamodule._target_}>")
datamodule: pl.LightningDataModule = hydra.utils.instantiate(
    cfg.data.datamodule, _recursive_=False
)

/home/gridsan/tmackey/cdvae/cdvae/pl_data/dataset.py:25: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  self.df = pd.read_csv(path)
/home/gridsan/tmackey/cdvae/cdvae/common/data_utils.py:672: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


using 54006 rows given a train_fraction of 1


  0%|          | 0/54006 [00:00<?, ?it/s]

/home/gridsan/tmackey/cdvae/cdvae/common/data_utils.py:615: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


In [7]:
datamodule.setup()

/home/gridsan/tmackey/cdvae/cdvae/pl_data/dataset.py:25: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  self.df = pd.read_csv(path)
/home/gridsan/tmackey/cdvae/cdvae/common/data_utils.py:672: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


using 54006 rows given a train_fraction of 1


  0%|          | 0/54006 [00:00<?, ?it/s]

using 8731 rows given a train_fraction of 1


  0%|          | 0/8731 [00:00<?, ?it/s]

using 8720 rows given a train_fraction of 1


  0%|          | 0/8720 [00:00<?, ?it/s]

In [8]:
train_loader = datamodule.train_dataloader()

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/torch_geometric/deprecation.py:13: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [9]:
list_of_idxs = []
list_of_batchs = []

In [10]:
for idx, batch in enumerate(train_loader):
    list_of_idxs.append(idx)
    list_of_batchs.append(batch)

In [11]:
index = 0 

In [12]:
def new_dataloader_batch_processor(batch): 
    batch_reserve = batch
    xrd_int = batch_reserve[1]
    xrd_loc = batch_reserve[2]
    atom_spec = batch_reserve[3]
    disc_sim_xrd = batch_reserve[4]
    batch = batch[0]
    return batch_reserve, xrd_int, xrd_loc, atom_spec, disc_sim_xrd, batch

In [13]:
list_of_exceptions = []

In [14]:
from tqdm import tqdm

In [16]:
start = len(list_of_batchs)//2

In [17]:
for index in tqdm(range(start,len(list_of_batchs))):
    try: 
        idx = list_of_idxs[index]
        batch = list_of_batchs[index]

        batch_reserve, xrd_int, xrd_loc, atom_spec, disc_sim_xrd, batch = new_dataloader_batch_processor(batch)

        batch_all_frac_coords = []
        batch_all_atom_types = []
        batch_frac_coords, batch_num_atoms, batch_atom_types = [], [], []
        batch_lengths, batch_angles = [], []


        self = model
        batch = batch.to("cuda:0")
        atom_spec = batch.num_atoms.to("cuda:0")
        model.to('cuda:0')

        mu, log_var, z = self.encode(batch, xrd_int, xrd_loc, atom_spec, disc_sim_xrd)

        z = z.to("cuda:0")

        kld_loss = self.kld_loss(mu, log_var)


        if self.use_composition_constraint: 
            gt_elements = atom_spec
        else: 
            gt_elements = None

        (pred_num_atoms, pred_lengths_and_angles, pred_lengths, pred_angles,
        pred_composition_per_atom) = self.decode_stats(
            z, batch.num_atoms, batch.lengths, batch.angles, teacher_forcing = True, gt_elements = atom_spec)

        # sample noise levels.
        noise_level = torch.randint(0, self.sigmas.size(0),
                                    (batch.num_atoms.size(0),),
                                    device=self.device)
        used_sigmas_per_atom = self.sigmas[noise_level].repeat_interleave(
            batch.num_atoms, dim=0)

        type_noise_level = torch.randint(0, self.type_sigmas.size(0),
                                        (batch.num_atoms.size(0),),
                                        device=self.device)
        used_type_sigmas_per_atom = (
            self.type_sigmas[type_noise_level].repeat_interleave(
                batch.num_atoms, dim=0))

        # add noise to atom types and sample atom types.
        pred_composition_probs = F.softmax(
            pred_composition_per_atom.detach(), dim=-1)
        atom_type_probs = (
            F.one_hot(batch.atom_types - 1, num_classes=MAX_ATOMIC_NUM) +
            pred_composition_probs * used_type_sigmas_per_atom[:, None])

        rand_atom_types = torch.multinomial(
            atom_type_probs, num_samples=1).squeeze(1) + 1

        # add noise to the cart coords
        cart_noises_per_atom = (
            torch.randn_like(batch.frac_coords) *
            used_sigmas_per_atom[:, None])
        cart_coords = frac_to_cart_coords(
            batch.frac_coords, pred_lengths, pred_angles, batch.num_atoms)
        cart_coords = cart_coords + cart_noises_per_atom
        noisy_frac_coords = cart_to_frac_coords(
            cart_coords, pred_lengths, pred_angles, batch.num_atoms)

        if self.decoder_dropout > 0.0:
            print("using decoder dropout")
            z = F.dropout(z, p=self.decoder_dropout, training=self.training)

        if self.type_fixing: 
            pred_cart_coord_diff, pred_atom_types = self.decoder(
            z, noisy_frac_coords, batch.atom_types, batch.num_atoms, 
            pred_lengths, pred_angles, gt_elements, dropout = self.decoder_dropout, is_training = True)
        else: 
            pred_cart_coord_diff, pred_atom_types = self.decoder(
            z, noisy_frac_coords, rand_atom_types, batch.num_atoms, 
            pred_lengths, pred_angles, gt_elements, dropout = self.decoder_dropout, is_training = True)


        if self.use_diffraction_loss:    #get the diffraction pattern from the prediction 
            decoded_xrd_loc, decoded_xrd_int = self.get_diffraction_pattern(pred_cart_coord_diff, pred_atom_types, batch.num_atoms, pred_lengths, pred_angles)

        # compute loss.
        num_atom_loss = self.num_atom_loss(pred_num_atoms, batch)

        lattice_loss = self.lattice_loss(pred_lengths_and_angles, batch)
        composition_loss = self.composition_loss(
            pred_composition_per_atom, batch.atom_types, batch)
        coord_loss = self.coord_loss(
            pred_cart_coord_diff, noisy_frac_coords, used_sigmas_per_atom, batch)
        type_loss = self.type_loss(pred_atom_types, batch.atom_types,
                                used_type_sigmas_per_atom, batch)
    except: 
        list_of_exceptions.append(batch)

  0%|          | 0/27003 [00:00<?, ?it/s]/home/gridsan/tmackey/cdvae/cdvae/common/data_utils.py:625: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
/home/gridsan/tmackey/cdvae/cdvae/common/data_utils.py:621: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
100%|██████████| 27003/27003 [19:16<00:00, 23.36it/s]


In [20]:
list_of_exceptions[0].atom_types

tensor([64, 64, 90, 90, 32, 32, 32, 32, 78, 78, 78, 78], device='cuda:0')

In [21]:
list_of_exceptions[1].atom_types

tensor([58, 64, 64, 49, 49, 49, 28, 28, 28], device='cuda:0')